In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
# KGAT_2ec3cbfb8183b69ded54729368abbf53
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

piotereks_glovec_pl_100d_path = kagglehub.dataset_download('piotereks/glovec-pl-100d')

print('Data source import complete.')


In [ ]:
all_words.index("wpisz")

In [ ]:
from pathlib import Path
from gensim.models import KeyedVectors
nn = 'fasttext_100_3_polish.bin'
vec_path=Path('datasets') / Path(nn)
print(vec_path, vec_path.exists())
print(str(vec_path))
word2vec = KeyedVectors.load(str(vec_path))
wv2 = word2vec

# loading but  wv.key_to_index does not work


In [ ]:
print(type(wv))
print(type(wv2))
# word2vec.index('wpisz')
wv=word2vec
print('wpisz' in wv.key_to_index)

In [ ]:
aa = wv.key_to_index.keys()
print(type(aa))
print(list(aa)[:10])
print(len(aa))

In [ ]:
print(type(wv))
# print(list(wv.keys()))
print(type(wv.key_to_index))
print(wv['wpisać'])
aa = wv.key_to_index.keys()
print(wv.key_to_index['wpisać'])
print(type(aa))
print(list(aa)[:10])
print(len(aa))


In [1]:
# Main fastttext load
import faiss
import numpy as np
from pathlib import Path
from gensim.models import FastText
# Load the gensim-saved FastText model (this .bin is a gensim pickle)
model_path = Path('datasets') / 'fasttext_100_3_polish.bin'
print('Loading gensim FastText model from', model_path)
model = FastText.load(str(model_path))  # looks the same as KeyedVectors.load 
word2vec = model.wv
wv = word2vec
print('Vocabulary size:', len(word2vec.index_to_key))
print(word2vec.similar_by_word('bierut', topn=20))

Loading gensim FastText model from datasets\fasttext_100_3_polish.bin


c:\Users\piote\.conda\envs\rap-generator\Lib\site-packages\gensim\utils.py:1460: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return _pickle.load(f, encoding='latin1')  # needed because loading from S3 doesn't support readline()


Vocabulary size: 2488306
[('bieruty', 0.9338290691375732), ('bieruta', 0.8975523114204407), ('gierut', 0.8936105966567993), ('bierutow', 0.8866199254989624), ('bieruła', 0.8502930998802185), ('bierutowianin', 0.8371340036392212), ('bierutowsko', 0.8317036628723145), ('bierutowski', 0.8239002823829651), ('bierutowa', 0.8169059753417969), ('bierutowo', 0.8112359642982483), ('korfanty', 0.8047600388526917), ('sieruta', 0.8035863041877747), ('bierun', 0.8023427724838257), ('bierutem', 0.8014403581619263), ('bieruń', 0.7978892922401428), ('kołajczyk', 0.797875702381134), ('bierutowic', 0.7930407524108887), ('bierl', 0.792570948600769), ('kędzierewicz', 0.7888312935829163), ('mościcki', 0.788409411907196)]


In [ ]:
word2vec.similar_by_vector

In [3]:

# --- 2. Prepare matrix ---
wv = word2vec
all_words = list(wv.key_to_index.keys())
all_vectors = np.array([wv[w] for w in all_words]).astype('float32')

# --- 3. Normalize once (IMPORTANT) ---
faiss.normalize_L2(all_vectors)

# --- 4. Build FAISS index (exact search) ---
d = all_vectors.shape[1]
index = faiss.IndexFlatIP(d)
index.add(all_vectors)


In [ ]:
print(list(word2vec.key_to_index.keys())[:100])

In [ ]:
# Diagnostic: check fastText installation, imports, and .bin load
import sys, subprocess, traceback
from pathlib import Path
print('Python executable:', sys.executable)

# Try common import names and report details
imported = False
for name in ('fasttext', 'fastText'):
    try:
        mod = __import__(name)
        print(f"Imported module '{name}' from: {getattr(mod, '__file__', 'built-in')}")
        print('module name attribute:', getattr(mod, '__name__', None))
        print('version:', getattr(mod, '__version__', 'unknown'))
        imported = True
        fasttext = mod
        break
    except Exception as e:
        print(f"Import {name} failed: {e}")

if not imported:
    print('\nfastText not importable in this kernel.')
    try:
        pip_output = subprocess.check_output([sys.executable, '-m', 'pip', 'show', 'fasttext']).decode(errors='ignore')
        print('\npip show fasttext:\n', pip_output)
    except Exception:
        pass
    try:
        pip_output = subprocess.check_output([sys.executable, '-m', 'pip', 'show', 'fasttext-wheel']).decode(errors='ignore')
        print('\npip show fasttext-wheel:\n', pip_output)
    except Exception:
        pass
    print('\nIf the package is installed but not importable, restart kernel or ensure notebook uses the same venv as shown above.')
else:
    # Try loading the .bin
    bin_path = Path('datasets') / 'fasttext_100_3_polish.bin'
    print('\nAttempting to load:', bin_path)
    try:
        model = fasttext.load_model(str(bin_path))
        words = [w for w in model.get_words() if not w.startswith('__label__')]
        print('Loaded OK — vocab size (no labels):', len(words))
        print('Nearest neighbors sample for "bierut":', model.get_nearest_neighbors('bierut')[:10])
    except Exception as e:
        print('fasttext.load_model error:')
        traceback.print_exc()
        print('\nIf load_model fails, try reinstalling a compatible wheel:')
        print('\npip uninstall fasttext fasttext-wheel -y')
        print('pip install --force-reinstall --no-cache-dir fasttext-wheel')
        print('Then restart the kernel and re-run this cell.')

In [ ]:
import os
import numpy as np
import faiss
from gensim.models import KeyedVectors

# Simple loader: focused behavior for .npy vector files
# Set these paths before running the cell
main_vec_file = 'datasets/fasttext_100_3_polish.bin.wv.vectors.npy'
main_vocab_file = 'datasets/fasttext_100_3_polish.bin.wv.vectors_vocab.npy'  # set to a .txt or .npy file containing words (one per line)

# from gensim.models import KeyedVectors

# if __name__ == '__main__':
#     word2vec = KeyedVectors.load("fasttext_100_3_polish.bin")
#     print(word2vec.similar_by_word("bierut"))

def _load_simple_npy_vectors(vec_path, vocab_path=None):
    vecs = np.load(vec_path, allow_pickle=True)
    if vecs.ndim != 2 or not np.issubdtype(vecs.dtype, np.floating):
        raise ValueError(f"Expected 2D float array in {vec_path}, got shape={vecs.shape}, dtype={vecs.dtype}")
    words = None
    # If user provided a vocab path, load it
    if vocab_path:
        if vocab_path.endswith('.npy'):
            words = [str(x) for x in np.load(vocab_path, allow_pickle=True).tolist()]
        else:
            with open(vocab_path, 'r', encoding='utf-8') as f:
                words = [ln.strip() for ln in f if ln.strip()]
    else:
        # Try a few common sibling filenames near the vectors file
        base = vec_path
        for suffix in ('.wv.vectors_vocab.npy', '.wv.vectors_vocab.txt', '.words.npy', '.words.txt', '.vocab.npy', '.vocab.txt'):
            cand = base.replace('.wv.vectors.npy', '') + suffix if '.wv.vectors.npy' in base else base[:-4] + suffix
            if os.path.exists(cand):
                if cand.endswith('.npy'):
                    words = [str(x) for x in np.load(cand, allow_pickle=True).tolist()]
                else:
                    with open(cand, 'r', encoding='utf-8') as f:
                        words = [ln.strip() for ln in f if ln.strip()]
                break
    if words is None:
        raise ValueError('Loaded .npy vectors but could not find a matching vocab file. Set `main_vocab_file` to a vocab .txt or .npy file.')
    if len(words) != vecs.shape[0]:
        raise ValueError(f'Vocab length ({len(words)}) does not match vectors rows ({vecs.shape[0]})')
    kv = KeyedVectors(vector_size=vecs.shape[1])
    kv.add_vectors(words, vecs.astype('float32'))
    return kv

# --- 1. Load embeddings (simple .npy support) ---
ext = os.path.splitext(main_vec_file)[1].lower()
if ext == '.npy':
    wv = _load_simple_npy_vectors(main_vec_file, vocab_path=main_vocab_file)
else:
    raise ValueError('This simple loader is configured only for .npy vector files. Set `main_vec_file` to a .npy file.')

# --- 2. Prepare matrix ---
all_words = list(wv.key_to_index.keys())
all_vectors = np.array([wv[w] for w in all_words]).astype('float32')

# --- 3. Normalize once (IMPORTANT) ---
faiss.normalize_L2(all_vectors)

# --- 4. Build FAISS index (exact search) ---
d = all_vectors.shape[1]
index = faiss.IndexFlatIP(d)
index.add(all_vectors)


In [ ]:
words[1]

In [ ]:
type(v_D_candidate)

In [ ]:
import numpy as np
# --- 2. Prepare matrix ---
all_words = list(wv.key_to_index.keys())
all_vectors = np.array([wv[w] for w in all_words]).astype('float32')

# --- 3. Normalize once (IMPORTANT) ---
faiss.normalize_L2(all_vectors)

# --- 4. Build FAISS index (exact search) ---
d = all_vectors.shape[1]
index = faiss.IndexFlatIP(d)
index.add(all_vectors)

In [16]:
import faiss
# --- 5a. Basic logic for word analogies  simple---
word_A = "king"
word_B = "queen"
word_C = "bucket"

word_A = "król"
word_B = "królowa"
word_C = "mężczyzna"
word_C = "wiadro"

# "tu wpisz swój tekst piosenki tutaj"

# word_A = "wpisz"
# word_B = "swój"
# word_C = "zajrzyj"

# word_A = "wpisz"
# word_B = "swój"
# word_C = "odbiec"



#  daleko odbiec

# "tam zajrzeć swój"

v_A = wv[word_A]
v_B = wv[word_B]
v_C = wv[word_C]

# Vector arithmetic
X_diff = v_B - v_A
v_D_candidate = v_C + X_diff

wv.similar_by_vector(v_D_candidate, topn=10)



[('wiadro', 0.7556779980659485),
 ('skraplarka', 0.7272395491600037),
 ('śrubówka', 0.7249431014060974),
 ('zmywarka', 0.7211113572120667),
 ('torebeczka', 0.7187470197677612),
 ('gąbeczka', 0.7176117300987244),
 ('kroplówka', 0.7121760845184326),
 ('żłobiarka', 0.7119558453559875),
 ('skrapiarka', 0.7104547619819641),
 ('blaszanka', 0.7075422406196594)]

In [ ]:
import faiss
# --- 5b. Basic logic for word analogies - more step by step---
word_A = "king"
word_B = "queen"
word_C = "bucket"

word_A = "król"
word_B = "królowa"
word_C = "mężczyzna"
# word_C = "wiadro"

# "tu wpisz swój tekst piosenki tutaj"

# word_A = "swój"
# word_B = "tekst"
# word_C = "swój"


# "tam zajrzeć swój"

v_A = wv[word_A]
v_B = wv[word_B]
v_C = wv[word_C]

# Vector arithmetic
X_diff = v_B - v_A
v_D_candidate = v_C + X_diff

# Normalize query vector
v_D_candidate = v_D_candidate.astype('float32')
faiss.normalize_L2(v_D_candidate.reshape(1, -1))




# --- 6. Search nearest neighbors ---
k = 25  # Increase k to account for filtered words
scores, indices = index.search(v_D_candidate.reshape(1, -1), k)

# --- 7. Output ---
print(f"Input: {word_C} + ({word_B} - {word_A})")

# Filter out input words (A, B, C) from the results
filtered_results_count = 0
for i in range(k):
    candidate_word = all_words[indices[0][i]]
    if candidate_word not in [word_A, word_B, word_C]:
        print(f"{filtered_results_count+1}: {candidate_word} (score={scores[0][i]:.4f})")
        filtered_results_count += 1
    if filtered_results_count >= 25: # Display top 5 unique results
        break


Input: mężczyzna + (królowa - król)
1: kobietka (score=0.7396)
2: kobieton (score=0.7304)
3: ninetka (score=0.7251)
4: kobieto» (score=0.7250)
5: -dziewczyna (score=0.7214)
6: dystka (score=0.7209)
7: brunetka (score=0.7185)
8: -dziewczynka (score=0.7142)
9: kobieity (score=0.7127)
10: osobniczka (score=0.7127)
11: kobietaw (score=0.7120)
12: nudystka (score=0.7091)
13: nastolatka (score=0.7089)
14: kobieta… (score=0.7088)
15: kobietš, (score=0.7085)
16: tworka (score=0.7079)
17: kobieta (score=0.7069)
18: 'dziewczyna (score=0.7056)
19: kobietv (score=0.7055)
20: kobiets (score=0.7052)
21: striptizerka (score=0.7049)
22: kobieta' (score=0.7039)
23: surferka (score=0.7037)
24: blondynka (score=0.7037)
25: kobietaz (score=0.7025)


In [7]:
print("in key_to_index:", "wpisz" in wv.key_to_index)
print("has_index_for:", wv.has_index_for("wpisz"))  # best check for real vocab index

vec = wv["wpisz"]  # may still work even if OOV
idx = wv.key_to_index.get("wpisz", None)
print("idx:", idx)
print("get_index:", wv.get_index("wpisz", default=-1))

in key_to_index: False
has_index_for: False
idx: None
get_index: -1


In [8]:
print(wv.key_to_index.get("wpisać", -1))
# wv.values_to_index.get("wpisz")
# type(wv.index_to_key[4127])
# wv.get_vector("Wpisać", None)
np.array_equal(wv["wpis"], wv["wpisz"])


4127


False

In [ ]:
def ft_get(kv, key, default=None):
    try:
        return kv[key]   # may return subword vector for OOV in FastText
    except KeyError:
        return default
    
words = wv.key_to_index.keys()

for i, w in enumerate(words):
    # idx = wv.key_to_index.get(w, -1)
    if np.array_equal(ft_get(wv, w), wv["wpisz"]):
        print(f"Found 'wpisz' at index {i} for word '{w}'")
        # break

In [22]:
type(wv)

gensim.models.fasttext.FastTextKeyedVectors

In [4]:
# Probably eighen calculation and space normalisation
import numpy as np
from scipy.linalg import eigh
import time

print("Starting vector processing...")
# ── 1. Load vectors (using pre-loaded wv) ───────────────────────────────────────────
# wv is already loaded by the previous cell (ZxB9XxVWSF6-)
words = list(wv.key_to_index.keys())

X = np.array([wv[w] for w in words]).astype(np.float32)
print(f"Using pre-loaded word vectors (wv) from Gensim. Initial word vectors (X) shape: {X.shape}")

# ── 2. Centre ─────────────────────────────────────────────────
print("Starting centering process...")
t0 = time.time()
mu  = X.mean(axis=0)                  # (dim,)
X_c = X - mu                           # (N, dim) — broadcast
print(f'Centering complete: {time.time()-t0:.2f}s, Centered matrix (X_c) shape: {X_c.shape}')

# ── 3. Covariance  (dim×dim, not N×N) ─────────────────────────
print("Starting covariance calculation...")
t0 = time.time()
cov = (X_c.T @ X_c) / len(X_c)        # (dim, dim)
print(f'Covariance complete: {time.time()-t0:.2f}s, Covariance matrix (cov) shape: {cov.shape}')

# ── 4. Eigendecomposition ─────────────────────────────────────
print("Starting eigendecomposition...")
t0 = time.time()
eigvals, U = eigh(cov)                 # eigvals (dim,), U (dim,dim)
eigvals = eigvals[::-1]               # flip to descending
U       = U[:, ::-1]
eigvals = np.maximum(eigvals, 1e-9)   # regularise near-zeros
print(f'Eigendecomposition complete: {time.time()-t0:.2f}s, Eigenvalues shape: {eigvals.shape}, Eigenvectors (U) shape: {U.shape}')

# ── 5. Build whitening matrix  W = Λ^{-½} Uᵀ ─────────────────
print("Building whitening matrix...")
W = U / np.sqrt(eigvals)               # (dim, dim), cols scaled
W = W.T                                # W shape (dim, dim)
print(f'Whitening matrix (W) built. Shape: {W.shape}')

# ── 6. Transform all vectors  x̃ = W(x−μ) ─────────────────────
print("Transforming all vectors (whitening)...")
t0 = time.time()
X_white = X_c @ W.T                   # (N, dim) — the bottleneck
print(f'Vector transformation complete: {time.time()-t0:.2f}s, Whitened vectors (X_white) shape: {X_white.shape}')

# ── 7. Save ───────────────────────────────────────────────────
print("Saving whitened vectors, whitening matrix, and mean...")
np.save('output-data/X_white.npy',  X_white)      # fast binary
np.save('output-data/W_matrix.npy', W)            # reuse for new words
np.save('output-data/mu.npy',       mu)
with open('output-data/words.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(words))
print("Save complete.")

# ── 8. Query new word at runtime ──────────────────────────────
print("Preparing for runtime queries...")
def whiten_vec(x_raw):
    # Ensure x_raw is a numpy array
    x_raw_np = np.asarray(x_raw, dtype=np.float32)
    return W @ (x_raw_np - mu)            # single O(d²) multiply

def cosine(a, b):
    a = a / np.linalg.norm(a)
    b = b / np.linalg.norm(b)
    return float(a @ b)


# example below has no practical meaning, just a test of the new space
word2idx = {w: i for i, w in enumerate(words)}
# Example usage:
# Check if words 'bank', 'river', 'loan' exist in the vocabulary before querying
test_words = ['bank', 'river', 'loan'] # These are English words, assuming Polish model has them or equivalent
available_test_words = [w for w in test_words if w in word2idx]

if len(available_test_words) >= 3:
    bank_vec  = X_white[word2idx['zamek']]
    river_vec = X_white[word2idx['rzeka']]
    loan_vec  = X_white[word2idx['pożyczka']]

    print(f"Cosine similarity between 'bank' and 'river': {cosine(bank_vec, river_vec)}")
    print(f"Cosine similarity between 'bank' and 'loan': {cosine(bank_vec, loan_vec)}")
else:
    print(f"Not all test words ({test_words}) found in vocabulary for similarity calculation. Available words: {list(word2idx.keys())[:10]}...")

print("Cell S7mm7qcZYOTz execution finished.")


Starting vector processing...
Using pre-loaded word vectors (wv) from Gensim. Initial word vectors (X) shape: (2488306, 100)
Starting centering process...
Centering complete: 0.35s, Centered matrix (X_c) shape: (2488306, 100)
Starting covariance calculation...
Covariance complete: 0.41s, Covariance matrix (cov) shape: (100, 100)
Starting eigendecomposition...
Eigendecomposition complete: 0.01s, Eigenvalues shape: (100,), Eigenvectors (U) shape: (100, 100)
Building whitening matrix...
Whitening matrix (W) built. Shape: (100, 100)
Transforming all vectors (whitening)...
Vector transformation complete: 0.37s, Whitened vectors (X_white) shape: (2488306, 100)
Saving whitened vectors, whitening matrix, and mean...
Save complete.
Preparing for runtime queries...
Cosine similarity between 'bank' and 'river': 0.004193891771137714
Cosine similarity between 'bank' and 'loan': -0.05547875165939331
Cell S7mm7qcZYOTz execution finished.


In [5]:
# --- 8. Update the original model with whitened vectors (in-place) ---
import copy
import numpy as np

# 1) Deep copy the original keyed vectors
wv_tweaked = copy.deepcopy(wv)

# 2) Replace only vocab vectors with whitened vectors (X_white)
if X_white.shape != (len(words), wv_tweaked.vector_size):
    raise ValueError(
        f"Shape mismatch: X_white={X_white.shape}, expected {(len(words), wv_tweaked.vector_size)}"
    )

# Map each word to its row index in the copied model to avoid ordering issues
row_idx = np.fromiter((wv_tweaked.key_to_index[w] for w in words), dtype=np.int64, count=len(words))
wv_tweaked.vectors[row_idx] = X_white.astype(np.float32, copy=False)

# Invalidate/recompute cached norms after direct vector mutation
wv_tweaked.fill_norms(force=True)



print(type(wv_tweaked))
print("Tweaked vectors shape:", wv_tweaked.vectors.shape)
print("Example token in tweaked vocab:", words[0], "->", wv_tweaked[words[0]][:5])

<class 'gensim.models.fasttext.FastTextKeyedVectors'>
Tweaked vectors shape: (2488306, 100)
Example token in tweaked vocab: tytuł -> [ 0.71713036  0.48925015  0.17974973  0.63235754 -3.798925  ]


In [ ]:
# Build search data from wv_tweaked
all_words_tweaked = list(wv_tweaked.key_to_index.keys())
all_vectors_tweaked = np.array([wv_tweaked[w] for w in all_words_tweaked]).astype('float32')
faiss.normalize_L2(all_vectors_tweaked)

d_tweaked = all_vectors_tweaked.shape[1]
index_tweaked = faiss.IndexFlatIP(d_tweaked)
index_tweaked.add(all_vectors_tweaked)

In [7]:
import faiss
print("[debug] Starting analogy on wv_tweaked...")

# --- 5. Basic logic for word analogies (wv_tweaked) ---
word_A = "king"
word_B = "queen"
word_C = "bucket"

word_A = "król"
word_B = "królowa"
word_C = "mężczyzna"
word_C = "wiadro"

# "tu wpisz swój tekst piosenki tutaj"

# word_A = "wpisz"
# word_B = "swój"
# word_C = "zajrzyj"

# word_A = "wpisz"
# word_B = "swój"
# word_C = "odbiec"

#  daleko odbiec

# "tam zajrzeć swój"

print(f"[debug] Input analogy: {word_C} + ({word_B} - {word_A})")
print("[debug] Fetching vectors from wv_tweaked...")
v_A = wv_tweaked[word_A]
print(f"[debug] v_A shape: {v_A.shape}")
v_B = wv_tweaked[word_B]
print(f"[debug] v_B shape: {v_B.shape}")
v_C = wv_tweaked[word_C]
print(f"[debug] v_C shape: {v_C.shape}")
print(f"[debug] Shapes: v_A={v_A.shape}, v_B={v_B.shape}, v_C={v_C.shape}")

# Vector arithmetic
print("[debug] Running vector arithmetic...")
X_diff = v_B - v_A
v_D_candidate = v_C + X_diff
print(f"[debug] Candidate norm: {float((v_D_candidate**2).sum()**0.5):.4f}")

print("[debug] Querying nearest neighbors...")
results = wv_tweaked.similar_by_vector(v_D_candidate, topn=10)
print(f"[debug] Retrieved {len(results)} results")
print("[debug] Done.")

results

[debug] Starting analogy on wv_tweaked...
[debug] Input analogy: wiadro + (królowa - król)
[debug] Fetching vectors from wv_tweaked...
[debug] v_A shape: (100,)
[debug] v_B shape: (100,)
[debug] v_C shape: (100,)
[debug] Shapes: v_A=(100,), v_B=(100,), v_C=(100,)
[debug] Running vector arithmetic...
[debug] Candidate norm: 25.7695
[debug] Querying nearest neighbors...
[debug] Retrieved 10 results
[debug] Done.


[('wiadro', 0.7457529306411743),
 ('wiadro…', 0.669316828250885),
 ('wiadrowa', 0.650314450263977),
 ('wiadro:', 0.6487915515899658),
 ('wiadrowni', 0.6418724656105042),
 ('wiadrownia', 0.6333823800086975),
 ('wiadrowy', 0.6180216073989868),
 ('wiadro,', 0.6165559887886047),
 ('wiadrowska', 0.6130061149597168),
 ('wiadrowo', 0.6070500016212463)]

In [ ]:
import faiss
import numpy as np

# --- 5. Basic logic for word analogies (wv_tweaked) ---
word_A = "king"
word_B = "queen"
word_C = "bucket"

word_A = "król"
word_B = "królowa"
word_C = "mężczyzna"
# word_C = "wiadro"

# "tu wpisz swój tekst piosenki tutaj"

# word_A = "swój"
# word_B = "tekst"
# word_C = "swój"



v_A = wv_tweaked[word_A]
v_B = wv_tweaked[word_B]
v_C = wv_tweaked[word_C]

# Vector arithmetic
X_diff = v_B - v_A
v_D_candidate = v_C + X_diff

# Normalize query vector
v_D_candidate = v_D_candidate.astype('float32')
faiss.normalize_L2(v_D_candidate.reshape(1, -1))

# --- 6. Search nearest neighbors ---
k = 25  # Increase k to account for filtered words
scores, indices = index_tweaked.search(v_D_candidate.reshape(1, -1), k)

# --- 7. Output ---
print(f"Input: {word_C} + ({word_B} - {word_A}) [wv_tweaked]")

# Filter out input words (A, B, C) from the results
filtered_results_count = 0
for i in range(k):
    candidate_word = all_words_tweaked[indices[0][i]]
    if candidate_word not in [word_A, word_B, word_C]:
        print(f"{filtered_results_count+1}: {candidate_word} (score={scores[0][i]:.4f})")
        filtered_results_count += 1
    if filtered_results_count >= 25:  # Display top 25 unique results
        break

In [ ]:
# # wv["wpisz"]
# # len(list(wv.key_to_index.keys()))
# # words[words.index("wpisz")]
# # word2idx["wpisz"]
# # len(words)  #  2488306
# print(len(wv))
# # print(type(word2idx))
print(type(wv))
aaa = wv["wpisz"]
words = list(wv.key_to_index.keys())
print(type(words))
bbb = words.index("wpisz")

In [11]:
# --- 5. Analogy Logic in Whitened Space   ver 1 --- to be verified if this works
word_A = "król"
word_B = "królowa"
word_C = "mężczyzna"
word_C = "wiadro"

# Get index for the words
idx_A = word2idx[word_A]
idx_B = word2idx[word_B]
idx_C = word2idx[word_C]

# Use pre-computed whitened vectors from X_white
v_A_white = X_white[idx_A]
v_B_white = X_white[idx_B]
v_C_white = X_white[idx_C]

# Vector arithmetic in whitened space
X_diff_white = v_B_white - v_A_white
v_D_candidate_white = v_C_white + X_diff_white

# --- 6. Search nearest neighbors in X_white ---
# Normalize candidate for cosine similarity
query_vec = v_D_candidate_white / np.linalg.norm(v_D_candidate_white)

# X_white is already normalized from previous cell execution steps if FAISS logic was used,
# but to be safe and independent of index state, we normalize a copy or ensure normalization:
X_white_norm = X_white / np.linalg.norm(X_white, axis=1, keepdims=True)

# Calculate all cosine similarities
similarities = X_white_norm @ query_vec

# Get top indices
k = 25
top_indices = np.argsort(similarities)[::-1]

# --- 7. Output ---
print(f"Input: {word_C} + ({word_B} - {word_A}) [Whitened Space]")

filtered_results_count = 0
for idx in top_indices:
    candidate_word = words[idx]
    if candidate_word not in [word_A, word_B, word_C]:
        print(f"{filtered_results_count+1}: {candidate_word} (score={similarities[idx]:.4f})")
        filtered_results_count += 1
    if filtered_results_count >= 25:
        break

Input: wiadro + (królowa - król) [Whitened Space]
1: m³ (score=20.7919)
2: gaz (score=20.4726)
3: 1  (score=20.3135)
4: za (score=19.2355)
5:  (score=17.5358)
6: woda (score=17.5029)
7: a (score=16.0654)
8:  g (score=15.4631)
9:  (score=14.6316)
10: sok (score=14.4521)
11: „ (score=13.9569)
12: i (score=13.7969)
13: dym (score=13.6827)
14: ] (score=13.5140)
15:  m (score=13.3621)
16: ! (score=12.5795)
17: łza (score=12.3218)
18: 2  (score=12.2754)
19: woń (score=12.1842)
20:   ˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙˙ěąá i     (score=12.0672)
21: olej (score=11.8889)
22: dno (score=11.7488)
23: m² (score=

# All below to verify

## Advanced FAISS Search (IVF Algorithm)

In [ ]:
import faiss
import numpy as np

# --- 5. Analogy Logic in Whitened Space ---
word_A = "król"
word_B = "królowa"
word_C = "wiadro"

idx_A, idx_B, idx_C = word2idx[word_A], word2idx[word_B], word2idx[word_C]

v_A_white = X_white[idx_A]
v_B_white = X_white[idx_B]
v_C_white = X_white[idx_C]

X_diff_white = v_B_white - v_A_white
v_D_candidate_white = v_C_white + X_diff_white

# --- 6. Advanced FAISS Search (IVF Algorithm) ---
print("Configuring FAISS IndexIVFFlat (Partitioned KNN search)...")
d_white = X_white.shape[1]

# 1. Normalization
X_white_norm = X_white.copy().astype('float32')
faiss.normalize_L2(X_white_norm)

# 2. Define the algorithm: IndexIVFFlat (Voronoi partitions)
# nlist is the number of clusters (cells)
nlist = int(np.sqrt(X_white_norm.shape[0]))
quantizer = faiss.IndexFlatIP(d_white) # Coarse quantizer
index_ivf = faiss.IndexIVFFlat(quantizer, d_white, nlist, faiss.METRIC_INNER_PRODUCT)

# 3. Train the index (Required for IVF algorithms)
print(f"Training IVF index on {X_white_norm.shape[0]} vectors with {nlist} clusters...")
index_ivf.train(X_white_norm)
index_ivf.add(X_white_norm)

# 4. Set nprobe (How many neighboring clusters to search)
index_ivf.nprobe = 10

# 5. Prepare query vector
query_vec = v_D_candidate_white.reshape(1, -1).astype('float32')
faiss.normalize_L2(query_vec)

# 6. Execute KNN search algorithm
k = 25
scores, indices = index_ivf.search(query_vec, k)

# --- 7. Output ---
print(f"\nInput: {word_C} + ({word_B} - {word_A}) [Whitened Space via IVF-KNN]")
filtered_results_count = 0
for i in range(k):
    idx = indices[0][i]
    if idx == -1: continue # Skip if search failed to find enough neighbors
    candidate_word = words[idx]
    if candidate_word not in [word_A, word_B, word_C]:
        print(f"{filtered_results_count+1}: {candidate_word} (score={scores[0][i]:.4f})")
        filtered_results_count += 1
    if filtered_results_count >= 25:
        break

In [ ]:
# --- 5. Analogy Logic in Whitened Space   ver 2 --- to be verified if this works
import faiss
import numpy as np

# --- 5. Analogy Logic in Whitened Space ---
word_A = "król"
word_B = "królowa"
word_C = "mężczyzna"
word_C = "wiadro"

# Retrieve indices from word2idx created in the whitening cell
idx_A, idx_B, idx_C = word2idx[word_A], word2idx[word_B], word2idx[word_C]

# Select the whitened vectors
v_A_white = X_white[idx_A]
v_B_white = X_white[idx_B]
v_C_white = X_white[idx_C]

# Compute the target candidate vector (C + B - A)
X_diff_white = v_B_white - v_A_white
v_D_candidate_white = v_C_white + X_diff_white

# --- 6. FAISS K-Nearest Neighbors Search ---
print("Searching for closest neighbors using FAISS...")
d_white = X_white.shape[1]

# We use IndexFlatIP (Inner Product) because it's equivalent to Cosine Similarity
# when the vectors are L2-normalized.
index_white = faiss.IndexFlatIP(d_white)

# Normalize all vectors to unit length
X_white_norm = X_white.copy().astype('float32')
faiss.normalize_L2(X_white_norm)

# Add vectors to the FAISS index
index_white.add(X_white_norm)

# Prepare the query: normalize the candidate vector
query_vec = v_D_candidate_white.reshape(1, -1).astype('float32')
faiss.normalize_L2(query_vec)

# Search for the k closest neighbors
k = 25
scores, indices = index_white.search(query_vec, k)

# --- 7. Results ---
print(f"\nInput analogy: {word_C} + ({word_B} - {word_A})")
print("Closest neighbors (excluding inputs):")

filtered_results_count = 0
for i in range(k):
    idx = indices[0][i]
    candidate_word = words[idx]
    if candidate_word not in [word_A, word_B, word_C]:
        print(f"{filtered_results_count+1}: {candidate_word} (score={scores[0][i]:.4f})")
        filtered_results_count += 1
    if filtered_results_count >= 25:
        break

In [19]:
word_A = "tu"
word_B = "wpisz"
word_C = "tam"

v_A = wv_tweaked[word_A]
v_B = wv_tweaked[word_B]
v_C = wv_tweaked[word_C]
X_diff = v_B - v_A
v_D_candidate = v_C + X_diff
results = wv_tweaked.similar_by_vector(v_D_candidate, topn=10)
print(f"Input: {word_C} + ({word_B} - {word_A}) [wv_tweaked]")
for i, (candidate_word, score) in enumerate(results):
    if candidate_word not in [word_A, word_B, word_C]:
        print(f"{i+1}: {candidate_word} (score={score:.4f})")

Input: tam + (wpisz - tu) [wv_tweaked]
1: pos�ugiwanie (score=0.4414)
2: wizards (score=0.4310)
3: wizard] (score=0.4238)
4: mewoli (score=0.4221)
5: postanowi'em (score=0.4198)
6: triśur (score=0.4171)
7: poshzgiwanie (score=0.4155)
8: „studiowania (score=0.4153)
9: rhaegar, (score=0.4147)
10: postanowi� (score=0.4131)


In [ ]:
#  kula
import numpy as np

# --- Parameters ---
# Using a fixed numeric value as requested
margin_value = 6
word_A = "król"
word_B = "królowa"
word_C = "mężczyzna"
word_C = "wiadro"


# "tu wpisz swój tekst piosenki tutaj"

word_A = "tu"
word_B = "wpisz"
word_C = "tam"


# "tam zajrzeć swój"

# --- 1. Prepare candidate ---
# Using variables from the previous whitening/index cells
idx_A, idx_B, idx_C = word2idx[word_A], word2idx[word_B], word2idx[word_C]
v_A_white = X_white[idx_A]
v_B_white = X_white[idx_B]
v_C_white = X_white[idx_C]

# Compute the target candidate vector (C + B - A)
v_D_candidate_white = v_C_white + (v_B_white - v_A_white)

# --- 2. Define the Hypercube Bounds ---
# The cube is +/- 0.01 absolute numeric value per dimension
lower_bounds = v_D_candidate_white - margin_value
upper_bounds = v_D_candidate_white + margin_value

print(f"Searching for words inside a {X_white.shape[1]}D-cube (!{margin_value}) around the candidate...")

# --- 3. Find vectors within bounds ---
# Check which vectors in X_white satisfy lower <= value <= upper for ALL dimensions
mask = np.all((X_white >= lower_bounds) & (X_white <= upper_bounds), axis=1)

# Get indices and compute distances, excluding input words
inside_indices = np.where(mask)[0]
results = []
for i in inside_indices:
    w = words[i]
    if w in [word_A, word_B, word_C]:
        continue
    dist = np.linalg.norm(X_white[i] - v_D_candidate_white)
    results.append((w, i, float(dist)))

# Sort by ascending distance (closest first)
results.sort(key=lambda t: t[2])

# --- 4. Display Results ---
print(f"Total words found inside the hypercube: {len(results)}")
print("--------------------------------------------------")
if len(results) > 0:
    for rank, (word, idx, dist) in enumerate(results, start=1):
        print(f"{rank}: {word} (Euclidean distance to center: {dist:.4f})")
else:
    print(f"No words found within the !{margin_value} range.")

KeyError: 'wpisz'

In [ ]:
import numpy as np

# --- Parameters ---
# Using a fixed numeric value as requested
margin_value = 6
word_A = "król"
word_B = "królowa"
word_C = "mężczyzna"
word_C = "wiadro"

# --- 1. Prepare candidate ---
# Using variables from the previous whitening/index cells
idx_A, idx_B, idx_C = word2idx[word_A], word2idx[word_B], word2idx[word_C]
v_A_white = X_white[idx_A]
v_B_white = X_white[idx_B]
v_C_white = X_white[idx_C]

# Compute the target candidate vector (C + B - A)
v_D_candidate_white = v_C_white + (v_B_white - v_A_white)

# --- 2. Define the Hypercube Bounds ---
# The cube is +/- 0.01 absolute numeric value per dimension
lower_bounds = v_D_candidate_white - margin_value
upper_bounds = v_D_candidate_white + margin_value

print(f"Searching for words inside a {X_white.shape[1]}D-cube (!{margin_value}) around the candidate...")

# --- 3. Find vectors within bounds ---
# Check which vectors in X_white satisfy lower <= value <= upper for ALL dimensions
mask = np.all((X_white >= lower_bounds) & (X_white <= upper_bounds), axis=1)

# Get indices and words
inside_indices = np.where(mask)[0]
results = [(words[i], i) for i in inside_indices if words[i] not in [word_A, word_B, word_C]]

# --- 4. Display Results ---
print(f"Total words found inside the hypercube: {len(results)}")
print("--------------------------------------------------")
for i, (word, idx) in enumerate(results):
    # Calculate Euclidean distance for reference
    dist = np.linalg.norm(X_white[idx] - v_D_candidate_white)
    print(f"{i+1}: {word} (Euclidean distance to center: {dist:.4f})")

if len(results) == 0:
    print(f"No words found within the !{margin_value} range.")

In [ ]:
import numpy as np

# Diagnostic Cell: Check data alignment and range
print(f"Shape of X_white: {X_white.shape}")
print(f"Number of words in word2idx: {len(word2idx)}")

# Check a specific word's whitened vector
try:
    test_word = 'król'
    idx = word2idx[test_word]
    vec = X_white[idx]
    print(f"\nWhitened vector for '{test_word}' (first 5 dims): {vec[:5]}")
    print(f"Mean of this vector: {np.mean(vec):.4f}")
    print(f"Std Dev of this vector: {np.std(vec):.4f}")

    # Check the range of the candidate vector
    v_D = v_C_white + (v_B_white - v_A_white)
    print(f"\nCandidate D (first 5 dims): {v_D[:5]}")

    # Check min/max across all vectors to understand the space scale
    print(f"Global Min in X_white: {np.min(X_white):.4f}")
    print(f"Global Max in X_white: {np.max(X_white):.4f}")

    # Debug the mask for a large margin
    temp_margin = 1.0
    low, high = v_D - temp_margin, v_D + temp_margin
    mask_test = np.all((X_white >= low) & (X_white <= high), axis=1)
    print(f"\nWords found with margin 1.0: {np.sum(mask_test)}")

except Exception as e:
    print(f"Error during diagnostics: {e}")

In [ ]:
# Per-dimension min/max: whitened and original
import numpy as np
try:
    min_per_dim_white = np.min(X_white, axis=0)
    max_per_dim_white = np.max(X_white, axis=0)
    min_per_dim_X = np.min(X, axis=0)
    max_per_dim_X = np.max(X, axis=0)
    for dim in range(min_per_dim_white.shape[0]):
        print(f"{dim+1}) white min={min_per_dim_white[dim]:.6f} ; white max={max_per_dim_white[dim]:.6f} ; X min={min_per_dim_X[dim]:.6f} ; X max={max_per_dim_X[dim]:.6f}")
except Exception as e:
    print('Error computing per-dim stats:', e)

## Sigma (σ) scaling — margin = number of standard deviations

In [ ]:
v_B_white - v_A_white

In [ ]:
import numpy as np

# --- Parameters ---
# Margin expressed as number of standard deviations (σ).
# After PCA-whitening each dimension has Var=1, so σ_per_dim ≈ 1.
# This makes the margin explicit and self-documenting.
margin_sigma = 6.0
word_A = "król"
word_B = "królowa"
word_C = "wiadro"
word_C = "mężczyzna"

# --- 1. Prep ---
idx_A, idx_B, idx_C = word2idx[word_A], word2idx[word_B], word2idx[word_C]
v_A_white = X_white[idx_A]
v_B_white = X_white[idx_B]
v_C_white = X_white[idx_C]
v_D_candidate_white = v_C_white + (v_B_white - v_A_white)

# --- 2. Per-dimension σ (should be ≈1 everywhere after whitening) ---
sigma_per_dim = np.std(X_white, axis=0)          # shape (dim,)
print(f"σ range across dims: {sigma_per_dim.min():.4f} – {sigma_per_dim.max():.4f}  (expected ≈1)")

# --- 3. Hypercube bounds ---
lower_bounds = v_D_candidate_white - margin_sigma * sigma_per_dim
upper_bounds = v_D_candidate_white + margin_sigma * sigma_per_dim

print(f"Searching inside a {X_white.shape[1]}D-cube (±{margin_sigma}σ per dimension)...")

# --- 4. Mask + distances ---
mask = np.all((X_white >= lower_bounds) & (X_white <= upper_bounds), axis=1)
inside_indices = np.where(mask)[0]
results = []
for i in inside_indices:
    w = words[i]
    if w in [word_A, word_B, word_C]:
        continue
    dist = np.linalg.norm(X_white[i] - v_D_candidate_white)
    results.append((w, i, float(dist)))
results.sort(key=lambda t: t[2])

# --- 5. Display ---
print(f"Total words found: {len(results)}")
print("--------------------------------------------------")
if results:
    for rank, (word, idx, dist) in enumerate(results, start=1):
        print(f"{rank}: {word} (Euclidean dist: {dist:.4f})")
else:
    print(f"No words found within ±{margin_sigma}σ range.")

##  IQR scaling — margin = number of inter-quartile ranges

In [ ]:
import numpy as np

# --- Parameters ---
# Margin expressed in IQR units.
# IQR = Q75 - Q25.  For a normal dist: IQR ≈ 1.349σ → 1.5 IQR ≈ 2.0σ.
# Robust to outliers because it uses quantiles, not variance.
margin_iqr = 6
word_A = "król"
word_B = "królowa"
word_C = "wiadro"

# --- 1. Prep ---
idx_A, idx_B, idx_C = word2idx[word_A], word2idx[word_B], word2idx[word_C]
v_A_white = X_white[idx_A]
v_B_white = X_white[idx_B]
v_C_white = X_white[idx_C]
v_D_candidate_white = v_C_white + (v_B_white - v_A_white)

# --- 2. Per-dimension IQR ---
q25 = np.percentile(X_white, 25, axis=0)         # shape (dim,)
q75 = np.percentile(X_white, 75, axis=0)
iqr_per_dim = q75 - q25
# Approximate σ equivalent for reference
sigma_equiv = iqr_per_dim / 1.349
print(f"IQR range across dims: {iqr_per_dim.min():.4f} – {iqr_per_dim.max():.4f}")
print(f"≈σ equivalent:         {sigma_equiv.min():.4f} – {sigma_equiv.max():.4f}")

# --- 3. Hypercube bounds ---
lower_bounds = v_D_candidate_white - margin_iqr * iqr_per_dim
upper_bounds = v_D_candidate_white + margin_iqr * iqr_per_dim

print(f"Searching inside a {X_white.shape[1]}D-cube (±{margin_iqr} IQR per dimension)...")

# --- 4. Mask + distances ---
mask = np.all((X_white >= lower_bounds) & (X_white <= upper_bounds), axis=1)
inside_indices = np.where(mask)[0]
results = []
for i in inside_indices:
    w = words[i]
    if w in [word_A, word_B, word_C]:
        continue
    dist = np.linalg.norm(X_white[i] - v_D_candidate_white)
    results.append((w, i, float(dist)))
results.sort(key=lambda t: t[2])

# --- 5. Display ---
print(f"Total words found: {len(results)}")
print("--------------------------------------------------")
if results:
    for rank, (word, idx, dist) in enumerate(results, start=1):
        print(f"{rank}: {word} (Euclidean dist: {dist:.4f})")
else:
    print(f"No words found within ±{margin_iqr} IQR range.")

## Raw percent-of-range scaling — margin = fraction of each dimension's full span

In [ ]:
import numpy as np

# --- Parameters ---
# Margin expressed as a fraction of each dimension's (max - min) range.
# E.g. 0.05 = ±5% of the full per-dimension span.
# Simple but sensitive to outliers — prefer σ/IQR when outliers exist.
margin_pct = 0.05
word_A = "król"
word_B = "królowa"
word_C = "wiadro"

# --- 1. Prep ---
idx_A, idx_B, idx_C = word2idx[word_A], word2idx[word_B], word2idx[word_C]
v_A_white = X_white[idx_A]
v_B_white = X_white[idx_B]
v_C_white = X_white[idx_C]
v_D_candidate_white = v_C_white + (v_B_white - v_A_white)

# --- 2. Per-dimension range ---
dim_min  = np.min(X_white, axis=0)               # shape (dim,)
dim_max  = np.max(X_white, axis=0)
dim_range = dim_max - dim_min
margin_abs = margin_pct * dim_range              # absolute margin per dim
print(f"Range (max-min) across dims: {dim_range.min():.4f} – {dim_range.max():.4f}")
print(f"Absolute margin ({margin_pct*100:.1f}%):  {margin_abs.min():.4f} – {margin_abs.max():.4f}")

# --- 3. Hypercube bounds ---
lower_bounds = v_D_candidate_white - margin_abs
upper_bounds = v_D_candidate_white + margin_abs

print(f"Searching inside a {X_white.shape[1]}D-cube (±{margin_pct*100:.1f}% of range per dimension)...")

# --- 4. Mask + distances ---
mask = np.all((X_white >= lower_bounds) & (X_white <= upper_bounds), axis=1)
inside_indices = np.where(mask)[0]
results = []
for i in inside_indices:
    w = words[i]
    if w in [word_A, word_B, word_C]:
        continue
    dist = np.linalg.norm(X_white[i] - v_D_candidate_white)
    results.append((w, i, float(dist)))
results.sort(key=lambda t: t[2])

# --- 5. Display ---
print(f"Total words found: {len(results)}")
print("--------------------------------------------------")
if results:
    for rank, (word, idx, dist) in enumerate(results, start=1):
        print(f"{rank}: {word} (Euclidean dist: {dist:.4f})")
else:
    print(f"No words found within ±{margin_pct*100:.1f}% range.")

## Ellipsoidal (Mahalanobis) search — Sigma variant

In [ ]:
# word2idx["wpisz"],
words.index("wpisz")

In [14]:
# diff = X_white - v_D_candidate_white              # (N, dim)  broadcast
# mahal_dist = np.sqrt(np.sum((diff / sigma_per_dim) ** 2, axis=1))  # (N,)
sigma_per_dim.shape

(100,)

In [ ]:
import numpy as np

# --- Parameters ---
# Ellipsoidal search: Mahalanobis distance using per-dimension σ.
# d_M(x) = sqrt( Σ_d ((x_d - c_d) / σ_d)² )
#
# WHY chi-squared quantile was WRONG:
#   χ²(d) describes ||x||² when x~N(0,I)  (distance from ORIGIN).
#   The candidate c = v_C + v_B - v_A is NOT at the origin.
#   Real dist²  ~ non-central χ²(d, λ=||c||²),  E[dist] = sqrt(d + ||c||²).
#   Fix: compute all distances empirically and use the coverage percentile directly.
coverage_pct = 0.0010   # top fraction of all words to include (0.10 = closest 10%)
word_A = "król"
word_B = "królowa"
word_C = "wiadro"

# "tu wpisz swoj tekst piosenki tutaj"

# word_A = "tu"
# word_B = "wpisać"
# word_C = "wiadro"

# --- 1. Prep ---
idx_A, idx_B, idx_C = word2idx[word_A], word2idx[word_B], word2idx[word_C]
v_A_white = X_white[idx_A]
v_B_white = X_white[idx_B]
v_C_white = X_white[idx_C]
v_D_candidate_white = v_C_white + (v_B_white - v_A_white)

# --- 2. Per-dimension σ + EMPIRICAL radius from coverage percentile ---
sigma_per_dim = np.std(X_white, axis=0)           # (dim,) — ≈1 after whitening
d = X_white.shape[1]
diff = X_white - v_D_candidate_white              # (N, dim)  broadcast
mahal_dist = np.sqrt(np.sum((diff / sigma_per_dim) ** 2, axis=1))  # (N,)

radius_sigma = float(np.percentile(mahal_dist, coverage_pct * 100))
print(f"d={d}, ||candidate||={np.linalg.norm(v_D_candidate_white):.2f}")
print(f"Empirical dist: min={mahal_dist.min():.2f}  median={np.median(mahal_dist):.2f}  max={mahal_dist.max():.2f}")
print(f"Coverage {coverage_pct:.0%}  →  empirical radius = {radius_sigma:.3f}σ")
print(f"σ range across dims: {sigma_per_dim.min():.4f} – {sigma_per_dim.max():.4f}  (expected ≈1 after whitening)")

# --- 3. Filter by radius ---
mask = mahal_dist <= radius_sigma
inside_indices = np.where(mask)[0]
results = []
for i in inside_indices:
    w = words[i]
    if w in [word_A, word_B, word_C]:
        continue
    results.append((w, i, float(mahal_dist[i])))
results.sort(key=lambda t: t[2])

# --- 4. Display ---
print(f"Ellipsoidal search (radius ≤ {radius_sigma:.3f}σ, coverage ≈{coverage_pct:.0%}): {len(results)} words found")
print("--------------------------------------------------")
if results:
    for rank, (word, idx, dist) in enumerate(results, start=1):
        print(f"{rank}: {word} (Mahalanobis dist: {dist:.4f}σ)")
else:
    print(f"No words found within {radius_sigma:.3f}σ radius.")

d=100, ||candidate||=25.77
Empirical dist: min=17.27  median=27.32  max=309.92
Coverage 0%  →  empirical radius = 24.393σ
σ range across dims: 0.9973 – 0.9991  (expected ≈1 after whitening)
Ellipsoidal search (radius ≤ 24.393σ, coverage ≈0%): 2488 words found
--------------------------------------------------
1: wiadro… (Mahalanobis dist: 19.6856σ)
2: wiadrowa (Mahalanobis dist: 20.1235σ)
3: wiadro: (Mahalanobis dist: 20.1508σ)
4: wiadro, (Mahalanobis dist: 20.3205σ)
5: wiadrowy (Mahalanobis dist: 20.6106σ)
6: wiadrownia (Mahalanobis dist: 20.6343σ)
7: wiadrowni (Mahalanobis dist: 20.6439σ)
8: wiadrowska (Mahalanobis dist: 20.8086σ)
9: wiadrowo (Mahalanobis dist: 20.8603σ)
10: „wiadro (Mahalanobis dist: 21.5330σ)
11: wiadra… (Mahalanobis dist: 21.8269σ)
12: wiaderko (Mahalanobis dist: 21.8867σ)
13: mydliny (Mahalanobis dist: 21.9454σ)
14: wiadraz (Mahalanobis dist: 21.9610σ)
15: bukłak (Mahalanobis dist: 22.0099σ)
16: strzykawka (Mahalanobis dist: 22.0484σ)
17: wiadrowie (Mahalanobis d

In [15]:
# Recompute exactly the Mahalanobis distances used earlier
sigma_per_dim = np.std(X_white, axis=0)
target = v_D_candidate_white  # or whichever target you used
diff = X_white - target
mahal_dist = np.sqrt(np.sum((diff / sigma_per_dim) ** 2, axis=1))

# Global minima
argmin = int(np.argmin(mahal_dist))
print("global min:", float(mahal_dist[argmin]), "index:", argmin, "word:", words[argmin])

# Show the 20 closest by Mahalanobis
for rank, i in enumerate(np.argsort(mahal_dist)[:20], start=1):
    print(f"{rank:2d}: {words[i]:20s}  dist={mahal_dist[i]:8.4f}")

# If 'wiadro' is in vocab, show its distance
if "wiadro" in words:
    iw = words.index("wiadro")
    print("wiadro index:", iw, "dist:", float(mahal_dist[iw]))
else:
    print("wiadro not in words")

# If your selection/results list came from a function that banned words,
# print the first returned result and its mahalanobis distance to compare:
print("first returned result from your function:", results[0] if 'results' in globals() else None)

global min: 17.274681091308594 index: 10156 word: wiadro
 1: wiadro                dist= 17.2747
 2: wiadro…               dist= 19.6856
 3: wiadrowa              dist= 20.1235
 4: wiadro:               dist= 20.1508
 5: wiadro,               dist= 20.3205
 6: wiadrowy              dist= 20.6106
 7: wiadrownia            dist= 20.6343
 8: wiadrowni             dist= 20.6439
 9: wiadrowska            dist= 20.8086
10: wiadrowo              dist= 20.8603
11: „wiadro               dist= 21.5330
12: wiadra…               dist= 21.8269
13: wiaderko              dist= 21.8867
14: mydliny               dist= 21.9454
15: wiadraz               dist= 21.9610
16: bukłak                dist= 22.0099
17: strzykawka            dist= 22.0484
18: wiadrowie             dist= 22.0512
19: kubeł-                dist= 22.0533
20: dzbanek               dist= 22.0787
wiadro index: 10156 dist: 17.274681091308594
first returned result from your function: ('wiadro…', np.int64(996874), 19.685636520385742)


## Song transfer using sigma Mahalanobis diffs (adaptive coverage)

In [ ]:
import numpy as np
import re
import unicodedata

# Build a tokenizer that keeps Polish letters and apostrophes.
def _tokenize_song(text):
    return re.findall(r"[^\W\d_]+(?:'[^\W\d_]+)?", text.lower(), flags=re.UNICODE)


def _ascii_key(token):
    # Fold diacritics: swój -> swoj
    return "".join(
        ch for ch in unicodedata.normalize("NFKD", token.lower())
        if not unicodedata.combining(ch)
    )


def _build_ascii_map(vocab_words):
    # Map ASCII-folded token -> preferred vocab word.
    # If many words collapse to same key, keep the shortest as a simple heuristic.
    mapping = {}
    for w in vocab_words:
        k = _ascii_key(w)
        if (k not in mapping) or (len(w) < len(mapping[k])):
            mapping[k] = w
    return mapping


ASCII_MAP = _build_ascii_map(words)


def _resolve_vocab_token(token):
    """
    Resolve token to vocabulary using:
    1) exact; 2) lowercase; 3) diacritic-folded match.
    """
    if token in word2idx:
        return token, "exact"
    t = token.lower()
    if t in word2idx:
        return t, "lower"
    k = _ascii_key(t)
    if k in ASCII_MAP:
        return ASCII_MAP[k], "ascii_fold"
    return None, "oov"


def _closest_candidate_sigma(target_vec, sigma_per_dim, banned_words=None, initial_coverage=1e-4, growth=10.0, max_coverage=0.5):
    """
    Find nearest candidate around target_vec using sigma-normalized Mahalanobis distance.
    Coverage expands adaptively until at least one valid candidate appears.
    """
    if banned_words is None:
        banned_words = set()

    diff = X_white - target_vec
    dist = np.sqrt(np.sum((diff / sigma_per_dim) ** 2, axis=1))

    coverage = float(initial_coverage)

    while coverage <= max_coverage:
        radius = float(np.percentile(dist, coverage * 100.0))
        candidate_idx = np.where(dist <= radius)[0]

        if candidate_idx.size > 0:
            ordered = candidate_idx[np.argsort(dist[candidate_idx])]
            for idx in ordered:
                w = words[idx]
                if w not in banned_words:
                    idx = int(idx)
                    return idx, float(dist[idx]), coverage, radius

        coverage *= growth

    # Fallback: nearest valid globally.
    ordered_all = np.argsort(dist)
    for idx in ordered_all:
        w = words[idx]
        if w not in banned_words:
            idx = int(idx)
            return idx, float(dist[idx]), max_coverage, float("nan")

    return None, float("nan"), max_coverage, float("nan")


def generate_song_by_diffs_sigma(song_text, new_start_word, initial_coverage=1e-4, growth=10.0, max_coverage=0.5):
    """
    Algorithm:
    1) For each source pair A=word_n, B=word_{n+1}, compute diff = v_B_white - v_A_white.
    2) Apply diff to current generated word vector: target = v_current + diff.
    3) Find nearest candidate with sigma Mahalanobis, expanding coverage if needed.
    4) Chosen word becomes next current word.

    OOV handling policy (always returns; avoids degenerate exact jumps):
    - Resolve source tokens via exact/lower/ascii-fold.
    - If unresolved, fallback to PREVIOUS RESOLVED SOURCE token (not current generated word).
    - This keeps diffs in the source-trajectory frame and prevents artifacts like
      target becoming exactly B when A is unresolved.
    """
    tokens = _tokenize_song(song_text)
    if len(tokens) < 2:
        raise ValueError("song_text must contain at least 2 words.")

    start_resolved, _ = _resolve_vocab_token(new_start_word)
    if start_resolved is None:
        raise ValueError(f"new_start_word '{new_start_word}' is not in vocabulary (even after ascii-fold).")

    sigma_per_dim = np.std(X_white, axis=0)
    sigma_per_dim = np.where(sigma_per_dim <= 1e-12, 1.0, sigma_per_dim)

    generated = [start_resolved]
    current_word = start_resolved
    debug_rows = []

    # Source-side anchor: last resolved source token.
    first_src_resolved, first_mode = _resolve_vocab_token(tokens[0])
    if first_src_resolved is None:
        first_src_resolved = start_resolved
        first_mode = "fallback_start"
    prev_source_resolved = first_src_resolved

    for i in range(len(tokens) - 1):
        src_A = tokens[i]
        src_B = tokens[i + 1]

        A_resolved, A_mode = _resolve_vocab_token(src_A)
        B_resolved, B_mode = _resolve_vocab_token(src_B)

        used_fallback = False

        if A_resolved is None:
            A_resolved = prev_source_resolved
            A_mode = "fallback_prev_source"
            used_fallback = True

        if B_resolved is None:
            # If B is OOV, stay near source continuity.
            B_resolved = A_resolved
            B_mode = "fallback_A_source"
            used_fallback = True

        idxA = word2idx[A_resolved]
        idxB = word2idx[B_resolved]
        idxCur = word2idx[current_word]

        diff_vec = X_white[idxB] - X_white[idxA]
        target_vec = X_white[idxCur] + diff_vec

        banned = {current_word}
        # Strict mode: when the source pair is fully resolved (status=ok),
        # never allow exact copy of B_resolved as the chosen output token.
        if not used_fallback:
            banned.add(B_resolved)
        idx_new, d_new, cov_used, radius_used = _closest_candidate_sigma(
            target_vec,
            sigma_per_dim,
            banned_words=banned,
            initial_coverage=initial_coverage,
            growth=growth,
            max_coverage=max_coverage,
        )

        if idx_new is None:
            generated.append(current_word)
            debug_rows.append({
                "step": i + 1,
                "A": src_A,
                "B": src_B,
                "A_resolved": A_resolved,
                "B_resolved": B_resolved,
                "A_mode": A_mode,
                "B_mode": B_mode,
                "status": "fallback_no_candidate",
                "chosen": current_word,
            })
            prev_source_resolved = B_resolved
            continue

        current_word = words[idx_new]
        generated.append(current_word)

        debug_rows.append({
            "step": i + 1,
            "A": src_A,
            "B": src_B,
            "A_resolved": A_resolved,
            "B_resolved": B_resolved,
            "A_mode": A_mode,
            "B_mode": B_mode,
            "status": "approx_oov_pair" if used_fallback else "ok",
            "chosen": current_word,
            "dist": d_new,
            "coverage_used": cov_used,
            "radius_used": radius_used,
        })

        prev_source_resolved = B_resolved

    new_song_text = " ".join(generated)
    return new_song_text, debug_rows

In [ ]:
# --- Usage ---
# Provide your source lyrics and a new starting word from vocabulary.
# Example below uses plain-ASCII Polish on purpose; resolver will try diacritic-fold mapping.
song_text = "tu wpisz swoj tekst piosenki tutaj"
new_start_word = "wiadro"

new_song_text, steps = generate_song_by_diffs_sigma(
    song_text=song_text,
    new_start_word=new_start_word,
    initial_coverage=1e-4,  # starts very tight
    growth=10.0,            # expands: 1e-4 -> 1e-3 -> 1e-2 -> ...
    max_coverage=0.5,       # safety cap before global nearest fallback
)

print("Generated target text:")
print("=" * 80)
print(new_song_text)
print("=" * 80)
print(f"Generated words: {len(new_song_text.split())}")

print("\nFirst steps preview:")
for row in steps[:10]:
    print(row)

## Ellipsoidal (Mahalanobis) search — Robust IQR variant

In [ ]:
import numpy as np

# --- Parameters ---
# Robust ellipsoidal search: Mahalanobis distance using per-dimension IQR.
# d_IQR(x) = sqrt( Σ_d ((x_d - c_d) / IQR_d)² )
#
# WHY chi-squared quantile was WRONG:
#   Same non-centrality issue as the sigma variant.
#   Fix: compute all IQR-distances empirically and use the coverage percentile.
coverage_pct = 0.10   # top fraction of all words to include (0.10 = closest 10%)
word_A = "król"
word_B = "królowa"
word_C = "wiadro"

# --- 1. Prep ---
idx_A, idx_B, idx_C = word2idx[word_A], word2idx[word_B], word2idx[word_C]
v_A_white = X_white[idx_A]
v_B_white = X_white[idx_B]
v_C_white = X_white[idx_C]
v_D_candidate_white = v_C_white + (v_B_white - v_A_white)

# --- 2. Per-dimension IQR + EMPIRICAL radius from coverage percentile ---
q25 = np.percentile(X_white, 25, axis=0)          # (dim,)
q75 = np.percentile(X_white, 75, axis=0)
iqr_per_dim = q75 - q25
sigma_equiv = iqr_per_dim / 1.349
d = X_white.shape[1]
diff = X_white - v_D_candidate_white               # (N, dim)  broadcast
iqr_dist = np.sqrt(np.sum((diff / iqr_per_dim) ** 2, axis=1))  # (N,)

radius_iqr = float(np.percentile(iqr_dist, coverage_pct * 100))
print(f"IQR range across dims: {iqr_per_dim.min():.4f} – {iqr_per_dim.max():.4f}")
print(f"≈σ equivalent:         {sigma_equiv.min():.4f} – {sigma_equiv.max():.4f}")
print(f"d={d}, ||candidate||={np.linalg.norm(v_D_candidate_white):.2f}")
print(f"Empirical IQR-dist: min={iqr_dist.min():.2f}  median={np.median(iqr_dist):.2f}  max={iqr_dist.max():.2f}")
print(f"Coverage {coverage_pct:.0%}  →  empirical radius = {radius_iqr:.3f} IQR-units")

# --- 3. Filter by radius ---
mask = iqr_dist <= radius_iqr
inside_indices = np.where(mask)[0]
results = []
for i in inside_indices:
    w = words[i]
    if w in [word_A, word_B, word_C]:
        continue
    results.append((w, i, float(iqr_dist[i])))
results.sort(key=lambda t: t[2])

# --- 4. Display ---
print(f"Robust ellipsoidal search (radius ≤ {radius_iqr:.3f} IQR, coverage ≈{coverage_pct:.0%}): {len(results)} words found")
print("--------------------------------------------------")
if results:
    for rank, (word, idx, dist) in enumerate(results, start=1):
        print(f"{rank}: {word} (IQR-Mahalanobis dist: {dist:.4f})")
else:
    print(f"No words found within {radius_iqr:.3f} IQR radius.")